# Week 1.1 — Pandas Transformations

**Target: 90–120 min. Real data: MovieLens Latest Small.**

**Practice rule:** use no AI assistance.

In [12]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next((p for p in [Path.cwd()] + list(Path.cwd().parents) if (p/"reference"/"dataset_loader.py").exists()), Path.cwd())
sys.path.insert(0,str(ROOT/"reference"))

from dataset_loader import load_checked_csv

M="https://raw.githubusercontent.com/smanihwr/ml-latest-small/master/movies.csv"
R="https://raw.githubusercontent.com/smanihwr/ml-latest-small/master/ratings.csv"
T="https://raw.githubusercontent.com/smanihwr/ml-latest-small/master/tags.csv"

movies=load_checked_csv("movielens_movies.csv",M,["movieId","title","genres"],9000)
ratings=load_checked_csv("movielens_ratings.csv",R,["userId","movieId","rating","timestamp"],100000)
tags=load_checked_csv("movielens_tags.csv",T,["userId","movieId","tag","timestamp"],3000)

ratings["datetime"]=pd.to_datetime(ratings["timestamp"],unit="s")

movies.head(10), ratings.head(10)

(   movieId                               title  \
 0        1                    Toy Story (1995)   
 1        2                      Jumanji (1995)   
 2        3             Grumpier Old Men (1995)   
 3        4            Waiting to Exhale (1995)   
 4        5  Father of the Bride Part II (1995)   
 5        6                         Heat (1995)   
 6        7                      Sabrina (1995)   
 7        8                 Tom and Huck (1995)   
 8        9                 Sudden Death (1995)   
 9       10                    GoldenEye (1995)   
 
                                         genres  
 0  Adventure|Animation|Children|Comedy|Fantasy  
 1                   Adventure|Children|Fantasy  
 2                               Comedy|Romance  
 3                         Comedy|Drama|Romance  
 4                                       Comedy  
 5                        Action|Crime|Thriller  
 6                               Comedy|Romance  
 7                           Adventur

### A. Audit/index/filter
Shape, dtypes, missing, duplicates, unique keys; `.loc`, `.iloc`, `.query`, sort, string year extraction.

In [13]:
for name, x in {"movies": movies, "ratings": ratings, "tags": tags}.items():
    print(name, x.shape)
    print(x.dtypes)
    print("missing", x.isna().sum())  # missing count
    print("duplicates", x.duplicated().sum())
    print("------------")

movies (9742, 3)
movieId    int64
title        str
genres       str
dtype: object
missing movieId    0
title      0
genres     0
dtype: int64
duplicates 0
------------
ratings (100836, 5)
userId                int64
movieId               int64
rating              float64
timestamp             int64
datetime     datetime64[us]
dtype: object
missing userId       0
movieId      0
rating       0
timestamp    0
datetime     0
dtype: int64
duplicates 0
------------
tags (3683, 4)
userId       int64
movieId      int64
tag            str
timestamp    int64
dtype: object
missing userId       0
movieId      0
tag          0
timestamp    0
dtype: int64
duplicates 0
------------


### B. Conditional counts
Users with >=100 ratings; among them avg <3.5. Movies >=50 ratings with avg >4. Return tidy tables.

In [ ]:
user_stats = ratings.groupby('userId').agg(
    num_ratings=('rating', 'size'),
    mean_rating=('rating', 'mean')
)
# using loc
active_user_low_avg = user_stats.loc[
    (user_stats.num_ratings >= 100) & (user_stats.mean_rating < 3.5)
]

movie_stats = ratings.groupby('movieId').agg(
    num_ratings=('rating', 'size'),
    mean_rating=('rating', 'mean')
)
# using query()
active_movie_high_avg = movie_stats.query(
    'num_ratings >= 50 & mean_rating > 4'
)

display(active_user_low_avg)
display(active_movie_high_avg)

pd.concat([active_user_low_avg, active_movie_high_avg], keys=["user", "movie"])


,num_ratings,mean_rating
userId,,
6,314,3.493631
7,152,3.230263
10,140,3.278571
15,135,3.448148
19,703,2.607397
...,...,...
600,763,2.991481
602,135,3.392593
604,100,3.480000


,num_ratings,mean_rating
movieId,,
50,204,4.237745
110,237,4.031646
111,104,4.105769
260,251,4.231076
293,133,4.018797
...,...,...
78499,55,4.109091
79132,143,4.066434
81845,58,4.043103


num_ratings  mean_rating
user  6               314     3.493631
      7               152     3.230263
      10              140     3.278571
      15              135     3.448148
      19              703     2.607397
...                   ...          ...
movie 78499            55     4.109091
      79132           143     4.066434
      81845            58     4.043103
      112852           59     4.050847
      116797           50     4.020000

[182 rows x 2 columns]

### C. groupby / named agg / transform
User count/mean/median/std/last date; within-user demean and z-score; movie summaries. Explain agg vs transform.

In [ ]:
user_summary = ratings.groupby("userId").agg(
    n=("rating","size"),
    mean=("rating","mean"),
    median=("rating","median"),
    sd=("rating","std"),
    last_date=("datetime","max"),
).reset_index()

ratings["user_mean"] = ratings.groupby("userId")["rating"].transform("mean")
ratings["user_sd"] = ratings.groupby("userId")["rating"].transform("std")
ratings["demeaned"] = ratings["rating"] - ratings["user_mean"]
ratings["z"] = ratings["demeaned"] / ratings["user_sd"]

### D. explode
Split `genres` on `|`, explode, genre counts, merge ratings, genre mean/count. Explain row multiplication.

In [ ]:
movie

### E. merge / join / cardinality
Safe ratings→movies merge with `validate=`; movie summary merge; repeat via indexed `.join`; demonstrate why a many-to-many merge can explode.

In [ ]:
# TODO

### F. concat and cross product
Split/concat ratings; cross join 3 rating buckets × 5 genres; predict row count first.

In [ ]:
# TODO

### G. pivot / pivot_table / melt
User×movie matrix for 30 most-rated movies, then melt back. Explain pivot vs pivot_table.

In [ ]:
# TODO

### H. rank / qcut / shift / rolling
Within-user rating percentile; movie popularity quintile; previous rating and rolling-10 mean after chronological sort.

In [ ]:
# TODO

In [ ]:
import pandas as pd

# Beispiel-DataFrame erstellen
df = pd.DataFrame({
    'A': [10, 20],
    'B': [30, 40]
}, index=['Zeile_1', 'Zeile_2'])

# Spalten UND Zeilen gleichzeitig umbenennen
df_neu = df.rename(
    columns={'A': 'Spalte_X', 'B': 'Spalte_Y'},
    index={'Zeile_1': 'Erste_Zeile'}
)

print(df_neu)

# Alle Spaltennamen in Kleinbuchstaben umwandeln
df = df.rename(columns=str.lower)

# Allen Index-Labels ein Präfix hinzufügen
df = df.rename(index=lambda x: f"ID_{x}")

`set_index()` improves data retrieval: Lets you look up rows quickly by their label using `.loc[...]` instead of filtering or searching by integer position.

### Flattening multi-index multi-value groupby-agg df:
```
df.groupby(["Col1", "Col2"]).agg(
    total_sales=("Col3", "sum"), avg_sales=("Col3", "mean")
).reset_index()
```